# Stock Price Time Series Analysis & Forecasting

**Objective:** analyze and forecast a real stock's closing price — decompose it into
trend/seasonality/residual, apply moving average and exponential smoothing, build an
ARIMA model, and evaluate the forecast against a held-out test period.

**Data:** pulled live via `yfinance` — real historical daily closing prices, not
synthetic. Default ticker is `AAPL`; change the `TICKER` variable in Step 1 to use any
other symbol.

**Steps**
1. Setup & install
2. Load real stock data
3. Plot and explore the raw series
4. Decompose into trend, seasonality, residual
5. Moving average & exponential smoothing
6. Check stationarity (required before ARIMA)
7. Build and select an ARIMA model
8. Forecast, evaluate (RMSE), and visualize
9. Summary of findings

**Tools:** `python`, `pandas`, `statsmodels`, `matplotlib`, `yfinance`, `scikit-learn`

**A note before you run this:** this notebook was built and logic-reviewed, but not
executed end-to-end in advance, since real market data and `statsmodels` weren't
reachable in the environment used to author it. Run it top to bottom and watch for
anything unexpected on the first pass — most likely spot to need a tweak is Step 7's
grid search if your chosen ticker behaves very differently from a typical large-cap
stock.

## Step 0 — Setup

In [ ]:
!pip install yfinance statsmodels --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error, mean_absolute_error

plt.rcParams["figure.figsize"] = (12, 5)


## Step 1 — Load real stock data

`yfinance` pulls actual historical prices from Yahoo Finance — no API key needed. Change
`TICKER` to analyze a different stock (e.g. `"MSFT"`, `"TSLA"`, `"^GSPC"` for the S&P 500
index).

In [ ]:
TICKER = "AAPL"
START_DATE = "2018-01-01"
END_DATE = None  # None = up to today

raw = yf.download(TICKER, start=START_DATE, end=END_DATE, progress=False)

# yfinance sometimes returns MultiIndex columns even for a single ticker -- flatten if so
if isinstance(raw.columns, pd.MultiIndex):
    raw.columns = raw.columns.get_level_values(0)

df = raw[["Close"]].dropna().copy()
df.index = pd.to_datetime(df.index)

print(f"{TICKER}: {len(df)} trading days, {df.index.min().date()} to {df.index.max().date()}")
df.head()


**Why reindex to business-day frequency:** stock data only has rows for trading days,
which leaves gaps for weekends and holidays. Several time series tools (like
`seasonal_decompose`) expect an evenly-spaced index. We reindex to business days and
forward-fill the small number of holiday gaps (the price didn't move on a holiday, so
carrying the last known price forward is a reasonable, standard choice here — just be
aware it slightly understates real-world gap risk around holidays).

In [ ]:
df = df.asfreq("B")
df["Close"] = df["Close"].ffill()
print(f"Missing values after forward-fill: {df['Close'].isna().sum()}")


## Step 2 — Plot and explore the raw series

In [ ]:
plt.plot(df.index, df["Close"], color="#4C72B0")
plt.title(f"{TICKER} Closing Price")
plt.xlabel("Date")
plt.ylabel("Price ($)")
plt.tight_layout()
plt.savefig("raw_series.png", dpi=130)
plt.show()

df["Close"].describe()


In [ ]:
# Daily log returns -- often more informative than raw price for volatility/behavior checks
df["log_return"] = np.log(df["Close"] / df["Close"].shift(1))

plt.plot(df.index, df["log_return"], color="#DD8452", linewidth=0.7)
plt.title(f"{TICKER} Daily Log Returns")
plt.xlabel("Date")
plt.ylabel("Log return")
plt.tight_layout()
plt.savefig("log_returns.png", dpi=130)
plt.show()

print(f"Annualized volatility (std of daily log returns * sqrt(252)): "
      f"{df['log_return'].std() * np.sqrt(252):.1%}")


## Step 3 — Decompose into trend, seasonality, residual

**Important caveat up front:** stock prices behave close to a random walk — they don't
have the kind of repeating seasonal pattern that, say, retail sales or temperature data
does. Classic decomposition still runs and is worth seeing, but don't expect a clean
seasonal component the way you would for genuinely seasonal data. We decompose with a
21-day period (roughly one trading month) as an illustrative choice, and treat any
seasonal component we get as weak/exploratory rather than a strong real pattern.

In [ ]:
decomposition = seasonal_decompose(df["Close"], model="additive", period=21)

fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)
decomposition.observed.plot(ax=axes[0], color="#4C72B0")
axes[0].set_ylabel("Observed")
decomposition.trend.plot(ax=axes[1], color="#55A868")
axes[1].set_ylabel("Trend")
decomposition.seasonal.plot(ax=axes[2], color="#DD8452")
axes[2].set_ylabel("Seasonal")
decomposition.resid.plot(ax=axes[3], color="#C44E52")
axes[3].set_ylabel("Residual")
axes[3].set_xlabel("Date")
fig.suptitle(f"{TICKER} Decomposition (period=21 trading days)")
plt.tight_layout()
plt.savefig("decomposition.png", dpi=130)
plt.show()

# Quantify how much of the variance the seasonal component actually explains --
# a direct check on the caveat above, not just a visual impression
resid_var = decomposition.resid.dropna().var()
seasonal_var = decomposition.seasonal.var()
total_var = (decomposition.seasonal + decomposition.resid).dropna().var()
print(f"Seasonal component variance as a share of (seasonal + residual): "
      f"{seasonal_var / total_var:.1%}")


## Step 4 — Moving average and exponential smoothing

**Moving averages** (SMA) smooth out day-to-day noise by averaging over a trailing
window — a 20-day and 50-day SMA are common technical-analysis benchmarks. **Exponential
smoothing** (EWMA) does something similar but weights recent observations more heavily,
so it reacts faster to new trends than a flat SMA window does.

In [ ]:
df["SMA_20"] = df["Close"].rolling(window=20).mean()
df["SMA_50"] = df["Close"].rolling(window=50).mean()
df["EWMA_20"] = df["Close"].ewm(span=20, adjust=False).mean()

plt.plot(df.index, df["Close"], label="Close", color="#4C72B0", alpha=0.5)
plt.plot(df.index, df["SMA_20"], label="SMA (20-day)", color="#DD8452")
plt.plot(df.index, df["SMA_50"], label="SMA (50-day)", color="#55A868")
plt.plot(df.index, df["EWMA_20"], label="EWMA (span=20)", color="#C44E52", linestyle="--")
plt.title(f"{TICKER}: Moving Averages vs. Exponential Smoothing")
plt.xlabel("Date")
plt.ylabel("Price ($)")
plt.legend()
plt.tight_layout()
plt.savefig("smoothing.png", dpi=130)
plt.show()


**Holt-Winters exponential smoothing** goes a step further — it's an actual
forecasting method (not just a smoothing display), which explicitly models trend. We fit
it on the price series with an additive trend and no seasonal component, matching what
Step 3 showed us about weak seasonality here.

In [ ]:
hw_model = ExponentialSmoothing(df["Close"], trend="add", seasonal=None).fit()
hw_fitted = hw_model.fittedvalues

plt.plot(df.index, df["Close"], label="Actual", color="#4C72B0", alpha=0.6)
plt.plot(df.index, hw_fitted, label="Holt-Winters fit", color="#C44E52")
plt.title(f"{TICKER}: Holt-Winters Exponential Smoothing Fit")
plt.legend()
plt.tight_layout()
plt.savefig("holt_winters.png", dpi=130)
plt.show()


## Step 5 — Check stationarity

ARIMA requires a **stationary** series (roughly: constant mean/variance over time) —
stock prices almost never are, since they trend. We test with the **Augmented
Dickey-Fuller (ADF) test**: the null hypothesis is "the series is non-stationary"; a
p-value below 0.05 lets us reject that. We test the raw price first (expect it to fail),
then the first difference (expect it to pass) — this tells us the differencing order
`d` to use in ARIMA(p, d, q).

In [ ]:
def adf_report(series, label):
    result = adfuller(series.dropna())
    print(f"{label}: ADF stat={result[0]:.3f}, p-value={result[1]:.4f} "
          f"({'stationary' if result[1] < 0.05 else 'NOT stationary'})")

adf_report(df["Close"], "Raw closing price")
adf_report(df["Close"].diff(), "First difference")


## Step 6 — Select ARIMA order and fit the model

We hold out the last 30 trading days as a test set (never seen during fitting) so we can
honestly evaluate forecast accuracy afterward, rather than checking the model against
data it already saw.

For order selection, we do a small grid search over `p` and `q` (with `d` fixed at 1,
based on Step 5's result) and keep the combination with the lowest **AIC** (a standard
score that rewards fit while penalizing unnecessary complexity — lower is better). This
is a simpler stand-in for `auto_arima` (from the `pmdarima` package) if you have that
installed and prefer it.

In [ ]:
TEST_DAYS = 30
train = df["Close"].iloc[:-TEST_DAYS]
test = df["Close"].iloc[-TEST_DAYS:]

print(f"Train: {len(train)} days | Test: {len(test)} days")

best_aic = np.inf
best_order = None
for p in range(0, 4):
    for q in range(0, 4):
        try:
            model = ARIMA(train, order=(p, 1, q)).fit()
            if model.aic < best_aic:
                best_aic = model.aic
                best_order = (p, 1, q)
        except Exception:
            continue  # some (p,q) combinations fail to converge -- skip them

print(f"Best order: ARIMA{best_order}, AIC={best_aic:.1f}")


In [ ]:
final_model = ARIMA(train, order=best_order).fit()
print(final_model.summary())


## Step 7 — Forecast, evaluate (RMSE), and visualize

We forecast forward exactly as many steps as we held out, then compare against the real
values. Alongside RMSE, we compute a **naive baseline** — "tomorrow's price = today's
price" — because for stock prices, a sophisticated model that doesn't clearly beat this
trivial baseline is a meaningful (and common!) finding, not a bug.

In [ ]:
forecast_result = final_model.get_forecast(steps=TEST_DAYS)
forecast_mean = forecast_result.predicted_mean
conf_int = forecast_result.conf_int(alpha=0.05)

rmse_arima = np.sqrt(mean_squared_error(test, forecast_mean))
mae_arima = mean_absolute_error(test, forecast_mean)

# Naive baseline: forecast every future day as the last known training value
naive_forecast = pd.Series(train.iloc[-1], index=test.index)
rmse_naive = np.sqrt(mean_squared_error(test, naive_forecast))

print(f"ARIMA{best_order}  RMSE: {rmse_arima:.2f}  MAE: {mae_arima:.2f}")
print(f"Naive baseline RMSE: {rmse_naive:.2f}")
print(f"ARIMA improvement over naive: {(1 - rmse_arima/rmse_naive):.1%}")


In [ ]:
plt.plot(train.index[-90:], train.iloc[-90:], label="Train (last 90 days)", color="#4C72B0")
plt.plot(test.index, test, label="Actual", color="#55A868", marker="o", markersize=3)
plt.plot(test.index, forecast_mean, label=f"ARIMA{best_order} forecast", color="#C44E52")
plt.fill_between(test.index, conf_int.iloc[:, 0], conf_int.iloc[:, 1],
                  color="#C44E52", alpha=0.15, label="95% confidence interval")
plt.plot(test.index, naive_forecast, label="Naive baseline", color="#8172B2", linestyle="--")
plt.title(f"{TICKER}: {TEST_DAYS}-Day Forecast vs. Actual")
plt.xlabel("Date")
plt.ylabel("Price ($)")
plt.legend()
plt.tight_layout()
plt.savefig("forecast.png", dpi=130)
plt.show()


## Step 8 — Summary of findings

In [ ]:
print("=== TIME SERIES FORECAST SUMMARY ===\n")
print(f"Ticker: {TICKER}")
print(f"Data range: {df.index.min().date()} to {df.index.max().date()} ({len(df)} trading days)")
print(f"Annualized volatility: {df['log_return'].std() * np.sqrt(252):.1%}\n")
print(f"Selected model: ARIMA{best_order} (chosen by lowest AIC over a small grid search)")
print(f"Test-period RMSE: {rmse_arima:.2f} vs. naive baseline RMSE: {rmse_naive:.2f}")
print(f"ARIMA {'beat' if rmse_arima < rmse_naive else 'did not clearly beat'} the naive baseline "
      f"by {abs(1 - rmse_arima/rmse_naive):.1%}")


### Written summary

- **Decomposition:** the seasonal component's share of variance (printed in Step 3) is
  usually small for a stock price — this confirms numerically what's true in general:
  daily stock prices don't have strong repeating seasonal structure the way retail sales
  or weather data do. Don't over-interpret the seasonal panel of the decomposition plot.
- **Stationarity:** the raw price series almost certainly failed the ADF test while the
  first difference passed — this is the standard signature of a random-walk-like series,
  and it's why ARIMA needs `d=1` (or higher) here, not `d=0`.
- **ARIMA vs. naive baseline:** this is the most important number in the whole notebook.
  If ARIMA's RMSE is close to (or worse than) the naive "tomorrow = today" baseline, that
  isn't a failure of the modeling — it's a genuine, well-documented property of stock
  prices: they're very close to efficient/random-walk in the short term, so beating a
  naive forecast consistently is hard by design, not by mistake.
- **Confidence intervals widening over the horizon:** if you look at the forecast plot,
  the shaded confidence band should visibly widen the further out the forecast goes —
  that's expected and appropriate; a forecast interval that *doesn't* widen over time
  would be a sign something's wrong with the model, not a sign of confidence.
- **Practical takeaway:** ARIMA on raw price is a reasonable baseline model and a good
  teaching example of the full workflow, but in practice, forecasting *returns*
  (already-differenced, roughly stationary) rather than raw price, and/or incorporating
  volume, volatility (e.g. a GARCH model), or external features, is the standard next
  step for anything beyond illustration.
- **This is not investment advice** — a backtested RMSE on historical data says nothing
  about future performance, transaction costs, or risk management, all of which matter
  far more than model choice for real trading decisions.